**Importing Library**

First we'll install category_encoders or ce library, this will help us in preprocessing and encoding process.

In [ ]:
pip install category_encoders

In [ ]:
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings("ignore")    

from matplotlib import pyplot
import seaborn as sns

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score, accuracy_score
from sklearn.ensemble import VotingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import category_encoders as ce
from imblearn.over_sampling import SMOTE

In [ ]:
aug_train = pd.read_csv('aug_train.csv')
aug_train.head()

**Dropping Uneeded Columns**

We will drop the uneeded columns based on our findings on the EDA part. Some of this columns is dropped due to it's main use is identification, and some due to findings in EDA regarding their effect to target column.

In [ ]:
aug_train.drop(columns=['enrollee_id','city','gender','company_size','company_type','training_hours'], axis = 1, inplace=True)

In [ ]:
aug_train.head()

**Handling missing values**

Next, we will try to handle the missing values. We can drop some of the Nan entries from columns which have a little missing data (~around 2%), and we will try to impute major_discipline with mode, since it's missing quite a lot of data.

In [ ]:
def count_percent(data):
  df_cols = pd.DataFrame({'Count Missing': data.isnull().sum(),
                        'Percent Missing': data.isnull().sum()*100/data.shape[0]})
  return df_cols

count_percent(aug_train)

In [ ]:
aug_train.dropna(subset=['enrolled_university', 'education_level','last_new_job','experience'], inplace = True)

In [ ]:
aug_train['major_discipline'].fillna(aug_train['major_discipline'].mode().iloc[0],inplace = True)


In [ ]:
count_percent(aug_train)

We see now that there are no more missing values in the column. This way we can proceed to encoding and scaling (If needed)

**Feature Engineering**

The first thing we will do is Encoding. In this notebook, we will mainly use two types of encoding. 
1. Ordinal Encoder: Used for Ordinal variable or variable which have a certain hierarchy. In this case we will use it to education_level, enrolled_university, experience, and last_new_job.
2. One Hot Encoder: Used for categorical variable which do not have a certain hierarchy or nominal variables. In this case, we will use it for relevent_experience and major_discipline.

Next there is scaling. Scaling is used to numerical feature so they have a small range. Some method will work the best with scaling done. But since we only have one more numerical value (city_development_index), and its range is already small (0 - 1), we will not be scaling any columns in this notebook.

In [ ]:
ordinal_mapping = [
    {'col':'education_level',
    'mapping':{'Primary School':1,'High School':2, 'Graduate':3, 'Masters':4, 'Phd':5}},
    {'col':'enrolled_university',
    'mapping':{'no_enrollment':1,'Part time course':2, 'Full time course':3}},
    {'col':'experience',
    'mapping':{'<1':1,'1':2, '2':3, '3':4, '4':5,'5':6,'6':7,'7':8,'8':9,'9':10,'10':11,'11':12,'12':13,'13':14,'14':15,'15':16,'16':17,'17':18,'18':19,'19':20,'20':21,'>20':22}},
    {'col':'last_new_job',
    'mapping':{'never':1,'1':2, '2':3, '3':4, '4':5,'>4':6}}    
    ]

ordinal_encoder = ce.OrdinalEncoder(cols = ['education_level','enrolled_university','experience','last_new_job'],mapping = ordinal_mapping)

In [ ]:
transformer = ColumnTransformer([
                                 ('one hot encoder',OneHotEncoder(),['relevent_experience','major_discipline']),
                                 ('binary encoder',ordinal_encoder,['education_level','enrolled_university','experience','last_new_job']),
], remainder = 'passthrough')

**Data Splitting**

Next we will split the aug_train data to train and test set.

In [ ]:
X = aug_train.drop(columns=['target'], axis = 1)
y = aug_train['target']

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, 
    y, 
    stratify = y,
    test_size = 0.2,)

Next, we will transformed (and fit) the train and test data so it will be encoded according to our settings.

In [ ]:
X_train_preprocessed = transformer.fit_transform(X_train_val)
X_test_preprocessed = transformer.transform(X_test)

In [ ]:
X_train_preprocessed = pd.DataFrame(X_train_preprocessed)
X_test_preprocessed = pd.DataFrame(X_test_preprocessed)
X_train_preprocessed

Since it will be confusing having no column name, we will try to get the features list and name it.

In [ ]:
transformer.transformers_[0][1].get_feature_names()

In [ ]:
transformer.transformers_[1][1].get_feature_names()

In [ ]:
features = list(['x0_Has relevent experience', 'x0_No relevent experience',
       'x1_Arts', 'x1_Business Degree', 'x1_Humanities', 'x1_No Major',
       'x1_Other', 'x1_STEM','education_level', 'enrolled_university', 'experience', 'last_new_job','city_development_index'])

In [ ]:
X_train_preprocessed.columns = features
X_test_preprocessed.columns = features
X_train_preprocessed

**Dropping One-hot Encoded Features**

One-hot encoded features only needs k-1 columns to represent the whole information. So we need to drop one column each. This will also prevent multicollinearity between the features.

In [ ]:
X_train_preprocessed.drop(columns=['x0_No relevent experience','x1_No Major'], axis = 1, inplace=True)
X_test_preprocessed.drop(columns=['x0_No relevent experience','x1_No Major'], axis = 1, inplace=True)

In [ ]:
X_train_preprocessed.head()

**Baseline Model**

Next we will try to build a model to finally predict who will leave and who will not. There are some methods used here and we will compare them and see which one gives us the best performance.

1. Decision Tree Classifier
2. Random Forest Classifier
3. Logistic Regressien
4. XGBClassifier
5. AdaBoostClassifier

We will try to evaluate these models with some metrics, but the main metric we're looking out for is Recall. Because the company wants to make sure that they get do not pick the candidate wrongly since it will lose the company money and time, and recall can represent this priority well.

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train_preprocessed, y_train_val)
y_pred = dt.predict(X_test_preprocessed)

In [ ]:
print('Recall Score:',recall_score(y_test,y_pred))
print('Precision Score:',precision_score(y_test,y_pred))
print('F1 Score:',f1_score(y_test,y_pred))
print('Accuracy Score:',accuracy_score(y_test,y_pred))

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_preprocessed, y_train_val)
y_pred = rf.predict(X_test_preprocessed)

In [ ]:
print('Recall Score:',recall_score(y_test,y_pred))
print('Precision Score:',precision_score(y_test,y_pred))
print('F1 Score:',f1_score(y_test,y_pred))
print('Accuracy Score:',accuracy_score(y_test,y_pred))

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_preprocessed, y_train_val)
y_pred = lr.predict(X_test_preprocessed)

In [ ]:
print('Recall Score:',recall_score(y_test,y_pred))
print('Precision Score:',precision_score(y_test,y_pred))
print('F1 Score:',f1_score(y_test,y_pred))
print('Accuracy Score:',accuracy_score(y_test,y_pred))

In [ ]:
xgb = XGBClassifier()
xgb.fit(X_train_preprocessed, y_train_val)
y_pred = xgb.predict(X_test_preprocessed)

In [ ]:
print('Recall Score:',recall_score(y_test,y_pred))
print('Precision Score:',precision_score(y_test,y_pred))
print('F1 Score:',f1_score(y_test,y_pred))
print('Accuracy Score:',accuracy_score(y_test,y_pred))

In [ ]:
ada = AdaBoostClassifier()
ada.fit(X_train_preprocessed, y_train_val)
y_pred = ada.predict(X_test_preprocessed)

In [ ]:
print('Recall Score:',recall_score(y_test,y_pred))
print('Precision Score:',precision_score(y_test,y_pred))
print('F1 Score:',f1_score(y_test,y_pred))
print('Accuracy Score:',accuracy_score(y_test,y_pred))

We see that mostly based on the metrics the models we use is nowehere near good enough. This can be caused by some factors, including imbalanced data. So we will fix this in the next step. So far XGBClassifier performs the best, but we will revisit all of these models performance later.

**Handling Imbalance**

To handle the imbalance problem, we will be using SMOTE to oversample the data.

In [ ]:
smote = SMOTE()
X_train_over, y_train_over = smote.fit_resample(X_train_preprocessed, y_train_val)

In [ ]:
X_train_preprocessed.shape

In [ ]:
X_over.shape

In [ ]:
y_train_val.value_counts()

In [ ]:
y_over.value_counts()

**Data splitting**


In [ ]:
X_train_over, X_val_over, y_train_over, y_val_over = train_test_split(
    X_over,
    y_over, 
    stratify = y_over,
    test_size = 0.20)

**Post-Oversampling Modelling**

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train_over, y_train_over)
y_pred = dt.predict(X_val_over)

In [ ]:
print('Recall Score:',recall_score(y_val_over,y_pred))
print('Precision Score:',precision_score(y_val_over,y_pred))
print('F1 Score:',f1_score(y_val_over,y_pred))
print('Accuracy Score:',accuracy_score(y_val_over,y_pred))

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_over, y_train_over)
y_pred = rf.predict(X_val_over)

In [ ]:
print('Recall Score:',recall_score(y_val_over,y_pred))
print('Precision Score:',precision_score(y_val_over,y_pred))
print('F1 Score:',f1_score(y_val_over,y_pred))
print('Accuracy Score:',accuracy_score(y_val_over,y_pred))

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_over, y_train_over)
y_pred = lr.predict(X_val_over)

In [ ]:
print('Recall Score:',recall_score(y_val_over,y_pred))
print('Precision Score:',precision_score(y_val_over,y_pred))
print('F1 Score:',f1_score(y_val_over,y_pred))
print('Accuracy Score:',accuracy_score(y_val_over,y_pred))

In [ ]:
xgb = XGBClassifier()
xgb.fit(X_train_over, y_train_over)
y_pred = xgb.predict(X_val_over)

In [ ]:
print('Recall Score:',recall_score(y_val_over,y_pred))
print('Precision Score:',precision_score(y_val_over,y_pred))
print('F1 Score:',f1_score(y_val_over,y_pred))
print('Accuracy Score:',accuracy_score(y_val_over,y_pred))

In [ ]:
ada = AdaBoostClassifier()
ada.fit(X_train_over, y_train_over)
y_pred = ada.predict(X_val_over)

In [ ]:
print('Recall Score:',recall_score(y_val_over,y_pred))
print('Precision Score:',precision_score(y_val_over,y_pred))
print('F1 Score:',f1_score(y_val_over,y_pred))
print('Accuracy Score:',accuracy_score(y_val_over,y_pred))

We see quite a significant increase! There's around 10% - 30% increase in the metrics across the board (Especially in Recall.) Right now it seems the best performing models are decision tree and random forest, followed by XGBClassifier.

**Voting Classifier**

Like it's name, a voting classifier can combine some of the classifier method we use by using a voting system. We will try using a voting classifier with some of the most high performing models which includes:

1. Decision Tree.
2. XGBClassifier
3. Random Forest.

In [ ]:
vc = VotingClassifier([('clf1',rf),('clf2',dt),('clf3',xgb)], voting = 'soft')
vc.fit(X_train_over, y_train_over)
y_pred = vc.predict(X_val_over)

In [ ]:
print('Recall Score:',recall_score(y_val_over,y_pred))
print('Precision Score:',precision_score(y_val_over,y_pred))
print('F1 Score:',f1_score(y_val_over,y_pred))
print('Accuracy Score:',accuracy_score(y_val_over,y_pred))

We see quite another increase. There's around 2 - 5% increase across the board. Right now the score hovers around 75% and that's around the minimum industry standard. Of course it could still be improved more using tuning - but we will not delve into it in this notebook. 


**Examining misprediction and Feature Importance**

Next we will try to see if our error is mostly caused by people who acts differently or have a certain pattern. We do this by combining prediction and the actual condition. Then we can see if the wrongly predicted have a different pattern than usual or not. 

We will also try to discover feature importance of each classifier used in the voting classifier. This may help us see which factors affects prediction the most and help us analyze why did the model did some prediction wrong.

In [ ]:
y_val_over = pd.DataFrame(y_val_over)
y_pred = pd.DataFrame(y_pred)
Test1 = pd.merge(X_val_over, y_val_over,left_index=True, right_index=True)
Test2 = pd.merge(Test1, y_pred,left_index=True, right_index=True)
Test3 = Test2[Test2['target'] != Test2[0]]
Test3

In [ ]:
importance = xgb.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
	print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

In [ ]:
importance = rf.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
	print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

In [ ]:
importance = dt.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
	print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

It seems the most important features includes city_development_index, experience, last new job, and education level. XGBoost also perceive relevent experience as extremely important.

If we see some of the table contents, we see that the wrongly predicted entries have different indicators based on the important columns. Like we see on the first record, the city_development_index and relevent experience is mostly alike with people who are not looking for jobs, but the education level is at 3 (graduate) and he still have a little experience which is alike to people who still look for jobs. Most of the records displayed have these patterns (mixed signals) so we concluded that most of the wrong prediction is because the entries do not have a mixed pattern.